In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# ROI Oscillation Comparison: `feedback_start` vs `cue_on`

This notebook computes ROI-level dominant oscillations using 2-40 Hz spectra with FOOOF-based aperiodic removal. It reuses the `Subject` caching utilities in `mv_regression_utils.py` and produces:

- average z-scored 1/f-removed PSD for each ROI and event
- histogram of dominant FOOOF peak frequencies across electrodes for each ROI
- paired t-tests of peak frequency between `feedback_start` and `cue_on` within ROI


In [ ]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from membandit._impl.multivariate import ROI_COLOR_MAP, OSCILLATION_EVENT_WINDOWS, OSCILLATION_FOOOF_DEFAULTS, plot_roi_oscillation_psd_summary, plot_roi_peak_frequency_histograms, plot_roi_property_deltas, plot_roi_property_summaries, run_roi_oscillation_comparison

sns.set_context('talk')
sns.set_style('whitegrid')


In [ ]:
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'

demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])].copy()

subjects = subj_df.MSSMCode.tolist()
rois = ['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC', 'dlPFC']

event_windows = dict(OSCILLATION_EVENT_WINDOWS)
fooof_kwargs = dict(OSCILLATION_FOOOF_DEFAULTS)

output_root = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Plots/roi_oscillation_comparison'
figure_dir = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit_ROI_OscillationComparison'
table_dir = os.path.join(output_root, 'tables')
os.makedirs(figure_dir, exist_ok=True)
os.makedirs(table_dir, exist_ok=True)

print(f'n_subjects={len(subjects)}')
print(f'output_root={output_root}')


In [ ]:
results = run_roi_oscillation_comparison(
    subjects=subjects,
    rois=rois,
    base_dir=base_dir,
    load_dir=load_dir,
    condition='Day1',
    freq_range=(2.0, 40.0),
    event_windows=event_windows,
    fooof_kwargs=fooof_kwargs,
    min_trials=10,
    n_jobs=-1,
    cache_mode='reuse',
    verbose=True,
)

electrode_psd_df = results['electrode_psd_df']
electrode_summary_df = results['electrode_summary_df']
matched_pairs_df = results['matched_pairs_df']
roi_psd_df = results['roi_psd_df']
property_stats_df = results['property_stats_df']
peak_stats_df = results['peak_stats_df']
audit_df = results['audit_df']

electrode_psd_df.to_csv(os.path.join(table_dir, 'electrode_psd_df.csv'), index=False)
electrode_summary_df.to_csv(os.path.join(table_dir, 'electrode_summary_df.csv'), index=False)
matched_pairs_df.to_csv(os.path.join(table_dir, 'matched_pairs_df.csv'), index=False)
roi_psd_df.to_csv(os.path.join(table_dir, 'roi_psd_df.csv'), index=False)
property_stats_df.to_csv(os.path.join(table_dir, 'property_stats_df.csv'), index=False)
peak_stats_df.to_csv(os.path.join(table_dir, 'peak_stats_df.csv'), index=False)
audit_df.to_csv(os.path.join(table_dir, 'audit_df.csv'), index=False)

property_stats_df.head(12)


In [ ]:
sns.barplot(data=electrode_psd_df[electrode_psd_df['event'] == 'cue_on'], 
x='roi', y='r_squared')
# sns.lineplot(data=electrode_psd_df[electrode_psd_df['event'] == 'cue_on'], x='frequency', y='psd_raw', hue='roi')

In [ ]:
# Load all
electrode_psd_df = pd.read_csv(os.path.join(table_dir, 'electrode_psd_df.csv'))
electrode_summary_df = pd.read_csv(os.path.join(table_dir, 'electrode_summary_df.csv'))
matched_pairs_df = pd.read_csv(os.path.join(table_dir, 'matched_pairs_df.csv'))
roi_psd_df = pd.read_csv(os.path.join(table_dir, 'roi_psd_df.csv'))
property_stats_df = pd.read_csv(os.path.join(table_dir, 'property_stats_df.csv'))
peak_stats_df = pd.read_csv(os.path.join(table_dir, 'peak_stats_df.csv'))
audit_df = pd.read_csv(os.path.join(table_dir, 'audit_df.csv'))


## Figure 5B:

In [ ]:
fig_psd, _ = plot_roi_oscillation_psd_summary(
    roi_psd_df=roi_psd_df,
    rois=rois,
    colors=ROI_COLOR_MAP,
    xscale='log',
)
fig_psd.savefig(os.path.join(figure_dir, 'roi_corrected_psd.pdf'), dpi=300, bbox_inches='tight')
plt.show()

fig_psd_raw, _ = plot_roi_oscillation_psd_summary(
    roi_psd_df=roi_psd_df,
    rois=rois,
    colors=ROI_COLOR_MAP,
    spectrum='raw_z',
    xscale='log'
)
fig_psd_raw.savefig(os.path.join(figure_dir, 'roi_uncorrected_normalized_psd.pdf'), dpi=300, bbox_inches='tight')
plt.show()

fig_summary, _ = plot_roi_property_summaries(
    electrode_summary_df=electrode_summary_df,
    rois=rois,
    colors=ROI_COLOR_MAP,
)
fig_summary.savefig(os.path.join(figure_dir, 'roi_property_summaries.png'), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# What is the statistic to look at the presence of theta in HPC and ACC? 

acc_theta_df = electrode_psd_df.loc[(electrode_psd_df['roi'].isin(['ACC'])) & (electrode_psd_df['frequency'] >= 3.5) & (electrode_psd_df['frequency'] < 8)]
hpc_theta_df = electrode_psd_df.loc[(electrode_psd_df['roi'].isin(['HPC'])) & (electrode_psd_df['frequency'] >= 3.5) & (electrode_psd_df['frequency'] < 8)]

In [ ]:
import statsmodels.formula.api as smf

acc_model = smf.mixedlm("psd_corrected_z ~ 1 + C(event)", groups='participant', 
data=acc_theta_df).fit()
print(acc_model.summary())

In [ ]:
hpc_model = smf.mixedlm("psd_corrected_z ~ 1 + C(event)", groups='participant', 
data=hpc_theta_df).fit()
print(hpc_model.summary())


In [ ]:
b


In [ ]:
if matched_pairs_df.empty:
    print('No matched electrodes with finite properties in both events; skipping delta figure.')
else:
    fig_delta, _ = plot_roi_property_deltas(
        matched_pairs_df=matched_pairs_df,
        rois=rois,
    )
    fig_delta.savefig(os.path.join(figure_dir, 'roi_property_deltas.png'), dpi=300, bbox_inches='tight')
    plt.show()

fig_hist, _ = plot_roi_peak_frequency_histograms(
    electrode_psd_df=electrode_psd_df,
    rois=rois,
    colors=ROI_COLOR_MAP,
)
fig_hist.savefig(os.path.join(figure_dir, 'roi_peak_frequency_histograms.png'), dpi=300, bbox_inches='tight')
plt.show()

property_stats_df
